# E46 --- a copula que a contagem nao escolhe

**A tentativa.** O capitulo dispensou a copula com um argumento de gosto: escolher errado erra
na cauda. Aqui o argumento vira consequencia de teorema. Com margens de contagem, o teorema de
Sklar garante que existe copula e **nao** garante que ela e unica --- e a contagem nao arbitra
entre os membros.

**O que se mede.**

1. a subcopula do par (sp500 x ibov), na grade das margens medidas;
2. os membros duplamente estocasticos que a estendem --- todos devolvem exatamente a mesma
   contagem no corte do capitulo;
3. no corte fundo, a banda entre o minimo e o maximo que os mesmos membros preveem, contra a
   taxa medida com a barra de blocos de 60 dias;
4. o controle: um par independente do mesmo tamanho, para mostrar que a banda e da contagem.

**Convencoes** (AGENTS.md §7 e §9): um experimento por caderno, parametros no topo marcados com
"brinque com", algoritmo em frevolab, resultado em lab/resultados/E46_identificacao.json,
figuras em .pdf e .png.

In [1]:
# <- brinque com: SERIE_A, SERIE_B, JANELA, CAUDA, CAUDAS_FUNDAS, NUS, BLOCOS, MUNDOS_CONTROLE, BLOCO, SEMENTE
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import frevolab
from frevolab import dados, dependencia, graficos, promessa, proporcao, volatilidade

SERIE_A = "sp500.csv"          # a primeira perna
SERIE_B = "ibov.csv"           # a segunda perna
JANELA = 252                   # o corte: um ano de pregoes
CAUDA = 0.05                   # a promessa do capitulo: um dia em vinte
CAUDAS_FUNDAS = (0.02, 0.01, 0.005)   # os cortes fundos do desenho
NUS = (2.0, 4.0, 8.0)          # os graus de liberdade da t declarada
BLOCOS = 400                   # as sortes de bloco para a barra da taxa funda
MUNDOS_CONTROLE = 30           # os pares independentes do controle
BLOCO = 60                     # o bloco que tem data
SEMENTE = 113                  # E40..E45 usam 107..112

retornos_a = volatilidade.retornos_log(dados.carregar_serie(SERIE_A))
retornos_b = volatilidade.retornos_log(dados.carregar_serie(SERIE_B))
comuns = retornos_a.index.intersection(retornos_b.index)
retornos_a, retornos_b = retornos_a.loc[comuns], retornos_b.loc[comuns]
rompe_a = promessa.violacoes(retornos_a, JANELA, CAUDA)
rompe_b = promessa.violacoes(retornos_b, JANELA, CAUDA)
limpa = rompe_a.notna() & rompe_b.notna()
rompe_a, rompe_b = rompe_a[limpa].astype(bool), rompe_b[limpa].astype(bool)

print("frevolab %s | %s: %d dias | %s: %d dias | comuns: %d"
      % (frevolab.VERSAO, SERIE_A, len(retornos_a), SERIE_B, len(retornos_b), len(comuns)))
print("rompimentos: %s %d (%.3f%%) | %s %d (%.3f%%)"
      % (SERIE_A, int(rompe_a.sum()), 100 * rompe_a.mean(),
         SERIE_B, int(rompe_b.sum()), 100 * rompe_b.mean()))

frevolab 0.1.0 | sp500.csv: 6450 dias | ibov.csv: 6450 dias | comuns: 6450
rompimentos: sp500.csv 315 (5.082%) | ibov.csv 315 (5.082%)


## A subcopula do par, e os membros que a estendem

A contagem assina os quatro cantos da subcopula --- os dois que cada margem assina sozinha, o
canto do um, e o canto de dia nenhum. A familia declarada constroi um subconjunto explicito das
extensoes duplamente estocasticas dessa subcopula: a gaussiana, uma t por grau de liberdade e a
mistura das bordas. Nenhuma bissecao escolhe membro --- todas casam a massa medida.

In [2]:
sub = dependencia.subcopula(rompe_a, rompe_b)
membros = dependencia.familia_compativel(sub["u_a"], sub["u_b"], sub["massa_retangulo"], nus=NUS)
residuo_max = max(m["residuo"] for m in membros)

print("dias %d | massa do retangulo %.5f (%d dias conjuntos) | canto de nenhum %.5f"
      % (sub["dias"], sub["massa_retangulo"], sub["contagem"]["juntos"], sub["canto_nenhum"]))
for m in membros:
    chave = ("nu %g" % m["nu"]) if m["familia"] == "t" else ("peso %.4f" % m["peso"] if m["familia"] == "mistura" else "")
    print("  %-10s %-10s correlacao %+.6f | canto %.7f | residuo %.2e"
          % (m["nome"], chave, m.get("correlacao", float("nan")), m["canto"], m["residuo"]))
print("residuo maximo da familia declarada: %.2e" % residuo_max)

dias 6198 | massa do retangulo 0.01823 (113 dias conjuntos) | canto de nenhum 0.91659
  gaussiana             correlacao +0.659800 | canto 0.9165860 | residuo 3.00e-15
  t_2        nu 2       correlacao +0.395617 | canto 0.9165860 | residuo 7.29e-14
  t_4        nu 4       correlacao +0.532180 | canto 0.9165860 | residuo 2.66e-14
  t_8        nu 8       correlacao +0.598900 | canto 0.9165860 | residuo 9.88e-15
  mistura    peso 0.3587 correlacao +nan | canto 0.9165860 | residuo 0.00e+00
residuo maximo da familia declarada: 7.29e-14


## A banda no corte do capitulo e no corte fundo

No corte do capitulo todos os membros devolvem a mesma taxa --- e a taxa medida. No corte fundo
os mesmos membros se abrem: a contagem estreita a familia e nao a escolhe.

In [3]:
niveis = (CAUDA,) + tuple(CAUDAS_FUNDAS)
ident = dependencia.conjunto_identificado(membros, niveis=niveis)

medido, barra = {}, {}
for q in niveis:
    qa, qb = retornos_a.quantile(q), retornos_b.quantile(q)
    conjunta = ((retornos_a <= qa) & (retornos_b <= qb)).astype(float)
    medido[q] = float(conjunta.mean())
    rng_q = np.random.default_rng(SEMENTE + int(round(1000 * q)))
    barra[q] = proporcao.barra_reamostrada(conjunta, 0.95, BLOCOS, rng=rng_q, bloco=BLOCO)

for q in niveis:
    d = ident[q]
    print("nivel %.3f | familia [%.5f, %.5f] razao %.2f | medido %.5f | barra [%.5f, %.5f]"
          % (q, d["minimo"], d["maximo"], d["razao"], medido[q], barra[q][0], barra[q][1]))

fundo = CAUDAS_FUNDAS[-1]
dentro = [q for q in niveis
          if ident[q]["minimo"] - 1e-12 <= medido[q] <= ident[q]["maximo"] + 1e-12]
print("cortes cuja taxa medida entra na banda: %d de %d" % (len(dentro), len(niveis)))

nivel 0.050 | familia [0.01786, 0.01794] razao 1.00 | medido 0.01783 | barra [0.01231, 0.02415]
nivel 0.020 | familia [0.00557, 0.00717] razao 1.29 | medido 0.00713 | barra [0.00280, 0.01371]
nivel 0.010 | familia [0.00232, 0.00359] razao 1.55 | medido 0.00403 | barra [0.00109, 0.00857]
nivel 0.005 | familia [0.00097, 0.00179] razao 1.85 | medido 0.00217 | barra [0.00031, 0.00483]
cortes cuja taxa medida entra na banda: 1 de 4


## O controle: um par independente, mesma rotina

Se a banda existe tambem sem dependencia nenhuma, entao a largura e da contagem --- e nao do
achar do par. O controle sorteia pares independentes do mesmo tamanho e mede a mesma razao.

In [4]:
rng_controle = np.random.default_rng(SEMENTE + 7)
razoes_controle, larguras_controle = [], []
for _ in range(MUNDOS_CONTROLE):
    fa = pd.Series(rng_controle.normal(0.0, 1.0, len(comuns)), index=comuns)
    fb = pd.Series(rng_controle.normal(0.0, 1.0, len(comuns)), index=comuns)
    ra = promessa.violacoes(fa, JANELA, CAUDA)
    rb = promessa.violacoes(fb, JANELA, CAUDA)
    ok = ra.notna() & rb.notna()
    s = dependencia.subcopula(ra[ok].astype(bool), rb[ok].astype(bool))
    fam = dependencia.familia_compativel(s["u_a"], s["u_b"], s["massa_retangulo"], nus=NUS)
    cj = dependencia.conjunto_identificado(fam, niveis=(fundo,))[fundo]
    razoes_controle.append(cj["razao"])
    larguras_controle.append(cj["maximo"] - cj["minimo"])

razao_controle = float(np.median(razoes_controle))
print("controle independente: razao mediana %.2f (min %.2f, max %.2f) | largura mediana %.5f"
      % (razao_controle, min(razoes_controle), max(razoes_controle),
         float(np.median(larguras_controle))))

controle independente: razao mediana 10.80 (min 7.51, max 14.34) | largura mediana 0.00023


In [5]:
# Figura 1: a taxa que cada membro preve, no corte do capitulo e no corte fundo.
fig, eixos = plt.subplots(1, 2, figsize=(9.4, 4.0))
nomes = [m["nome"] for m in membros]
predito_raso = [ident[CAUDA]["taxas"][n] for n in nomes]
predito_fundo = [ident[fundo]["taxas"][n] for n in nomes]

eixos[0].bar(nomes, 100 * np.array(predito_raso), color="#1f4e79")
eixos[0].axhline(100 * medido[CAUDA], color="#b03a2e", ls="--", lw=1.4,
                 label="a taxa medida: %.3f%%" % (100 * medido[CAUDA]))
eixos[0].set_title("no corte do capitulo (%.0f%%)" % (100 * CAUDA))
eixos[0].set_ylabel("taxa conjunta (%)")
eixos[0].legend(fontsize=8)

eixos[1].bar(nomes, 100 * np.array(predito_fundo), color="#c78f2c")
eixos[1].axhline(100 * ident[fundo]["minimo"], color="#555555", ls=":", lw=1.2,
                 label="minimo da familia: %.3f%%" % (100 * ident[fundo]["minimo"]))
eixos[1].axhline(100 * ident[fundo]["maximo"], color="#555555", ls="-.", lw=1.2,
                 label="maximo da familia: %.3f%%" % (100 * ident[fundo]["maximo"]))
eixos[1].errorbar([len(nomes) - 0.5], [100 * medido[fundo]],
                  yerr=[[100 * (medido[fundo] - barra[fundo][0])],
                        [100 * (barra[fundo][1] - medido[fundo])]],
                  fmt="o", color="#b03a2e", capsize=4, label="medido, com barra de blocos")
eixos[1].set_title("no corte fundo (%.1f%%)" % (100 * fundo))
eixos[1].legend(fontsize=8)

for eixo in eixos:
    eixo.tick_params(axis="x", rotation=20, labelsize=8)
fig.suptitle("a familia que a contagem admite: colada no raso, aberta no fundo", fontsize=10)
fig.tight_layout()
graficos.salvar(fig, "E46_identificacao", 1)
plt.close(fig)
print("figura E46_identificacao_1 salva")

figura E46_identificacao_1 salva


In [6]:
# Figura 2: a banda da familia contra o nivel do corte, com as bordas de Frechet.
fig, eixo = plt.subplots(figsize=(8.6, 4.2))
eixo.plot([100 * q for q in niveis], [100 * ident[q]["minimo"] for q in niveis], "o-",
          color="#1f4e79", label="minimo da familia")
eixo.plot([100 * q for q in niveis], [100 * ident[q]["maximo"] for q in niveis], "s-",
          color="#c78f2c", label="maximo da familia")
eixo.errorbar([100 * q for q in niveis], [100 * medido[q] for q in niveis],
              yerr=[[100 * (medido[q] - barra[q][0]) for q in niveis],
                    [100 * (barra[q][1] - medido[q]) for q in niveis]],
              fmt="D", color="#b03a2e", capsize=4, label="medido, com barra de blocos")

produto, minimo_margens = [], []
for q in niveis:
    p_a, p_b = sub["taxa_a"] * (q / CAUDA), sub["taxa_b"] * (q / CAUDA)
    produto.append(100 * p_a * p_b)
    minimo_margens.append(100 * min(p_a, p_b))
eixo.plot([100 * q for q in niveis], produto, ":", color="#555555", label="produto das margens")
eixo.plot([100 * q for q in niveis], minimo_margens, "-.", color="#555555", label="minimo das margens")
eixo.set_xscale("log")
eixo.set_xlabel("nivel do corte (%)")
eixo.set_ylabel("taxa conjunta (%)")
eixo.set_title("a banda que a contagem deixa em aberto", fontsize=10)
eixo.legend(fontsize=8)
fig.tight_layout()
graficos.salvar(fig, "E46_identificacao", 2)
plt.close(fig)
print("figura E46_identificacao_2 salva")

figura E46_identificacao_2 salva


## Leitura visual das figuras

**Declarada contra os .png depois da execucao** (AGENTS.md §9): a leitura se declara aqui, e o
criterio de frescor e o hash das celulas de codigo.

O que as legendas do capitulo afirmam, e a leitura tem de conferir nos .png:

1. **Figura 1**: no painel do corte do capitulo, as cinco barras da familia na mesma altura, com
   a linha da taxa medida sobre elas; no painel do corte fundo, as barras **abertas**, com os
   tracos do minimo e do maximo separados e o ponto medido com a barra de blocos entre eles.
2. **Figura 2**: as duas bordas da familia afunilando com o nivel, os pontos medidos dentro da
   banda em todos os niveis, e as bordas de Frechet por fora da banda.

In [7]:
# O resultado: um objeto por grandeza, em portugues, para o livro citar por comando.
resultado = {
    "familia_dias": int(sub["dias"]),
    "familia_membros": len(membros),
    "familia_residuo_max": float(residuo_max),
    "familia_massa_pct": round(100 * sub["massa_retangulo"], 4),
    "familia_conjuntos": int(sub["contagem"]["juntos"]),
    "familia_fundo_pct": round(100 * fundo, 3),
    "familia_fundo_min_pct": round(100 * ident[fundo]["minimo"], 4),
    "familia_fundo_max_pct": round(100 * ident[fundo]["maximo"], 4),
    "familia_fundo_razao": round(float(ident[fundo]["razao"]), 3),
    "familia_fundo_medido_pct": round(100 * medido[fundo], 4),
    "familia_fundo_barra_pct": round(100 * (barra[fundo][1] - medido[fundo]), 4),
    "familia_cortes_dentro": int(len(dentro)),
    "familia_cortes": int(len(niveis)),
    "familia_controle_razao": round(razao_controle, 3),
    "familia_controle_largura_pct": round(100 * float(np.median(larguras_controle)), 4),
    "familia_peso_mistura_pct": round(100 * float([m for m in membros if m["familia"] == "mistura"][0]["peso"]), 2),
}
caminho = Path("lab/resultados/E46_identificacao.json")
caminho.parent.mkdir(parents=True, exist_ok=True)
caminho.write_text(json.dumps(resultado, ensure_ascii=False, indent=1), encoding="utf-8")
print(json.dumps(resultado, ensure_ascii=False, indent=1))

{
 "familia_dias": 6198,
 "familia_membros": 5,
 "familia_residuo_max": 7.294165271787278e-14,
 "familia_massa_pct": 1.8232,
 "familia_conjuntos": 113,
 "familia_fundo_pct": 0.5,
 "familia_fundo_min_pct": 0.0971,
 "familia_fundo_max_pct": 0.1794,
 "familia_fundo_razao": 1.848,
 "familia_fundo_medido_pct": 0.2171,
 "familia_fundo_barra_pct": 0.2658,
 "familia_cortes_dentro": 1,
 "familia_cortes": 4,
 "familia_controle_razao": 10.798,
 "familia_controle_largura_pct": 0.0227,
 "familia_peso_mistura_pct": 35.87
}
